## 5.4 接入建链仿真

在上一节中，我们学习了六阶段接入建链流程。本节通过 SleNode 模拟广播者与发起者的完整接入握手，逐 SNR 统计接入成功率，观察 AWGN 噪声对建链各阶段的影响。

本节学习大纲如下：

- SleNode 建链过程拆解
- 各阶段成功率随 SNR 变化
- 数据往返成功率与整体成功率对比

### 本实验涉及的关键文件

```
src/nearlink_sdr/
├── node.py               <- SleNode: 完整节点模型 (广播/扫描/连接/收发)
├── mac/
│   ├── access.py          <- run_access_procedure: 六阶段接入流程
│   ├── link_manager.py    <- 链路状态机 (8 状态)
│   └── scheduler.py       <- ScheduleManager: 调度器配置
├── phy/
│   └── channel.py         <- ChannelModel: AWGN 加噪
└── sim/
    └── link_sim.py        <- sim_access_scheduled_link: 接入成功率扫描
```

---

### 1. 单次接入建链演示

创建 G 节点和 T 节点，执行完整的广播→扫描→连接→接受→数据交换流程：

In [ ]:
import sys
sys.path.insert(0, "../src")
import numpy as np
from nearlink_sdr.mac.link_manager import Role
from nearlink_sdr.node import NodeConfig, NodeRole, NodeState, SleNode

g_addr = b"\x01\x02\x03\x04\x05\x06"
t_addr = b"\x0A\x0B\x0C\x0D\x0E\x0F"

g = SleNode(config=NodeConfig(
    address=g_addr, role=NodeRole.G_NODE,
    frame_type=2, mcs_index=0, max_retransmit=0))
t = SleNode(config=NodeConfig(
    address=t_addr, role=NodeRole.T_NODE,
    frame_type=2, mcs_index=0, max_retransmit=0))

# 阶段 (a): 广播
adv = g.start_advertising()
print(f"1. G broadcast: {'OK' if adv else 'FAIL'} (state={g.state.name})")

# 阶段 (b): 扫描 + 接入请求
t.start_scanning()
print(f"2. T scanning (state={t.state.name})")
t.connect(g_addr)

# 阶段 (c)-(d): 接受 + 响应
g.accept_connection(t_addr, Role.G_NODE)
# 通过检查双方 peer_address 是否互配来判断连接成功
linked = t.peer_address == g_addr and g.peer_address == t_addr
scheduler_ok = 0 in g.scheduler.event_schedulers
print(f"3. Link established: {'OK' if linked else 'FAIL'}")
print(f"   G state={g.state.name}, T state={t.state.name}, scheduler={'OK' if scheduler_ok else 'FAIL'}")

# 阶段 (e): 数据交换
payload = b"hello_sle_access"
g.send(payload)
tx = g.transmit()
print(f"4. Data TX: {'OK' if tx.iq is not None else 'FAIL'}")

# 断开
g.disconnect()
print(f"5. Disconnect: G state={g.state.name}")

### 关键函数说明

上述演示中涉及的函数按阶段分为三组。从调用链来看，一次完整的数据交换走的是：

```
start_advertising() → start_scanning() → connect() → accept_connection()
→ send() → transmit() → AWGN → receive()
```

下面逐一拆解每个函数的内部逻辑和嵌套的关键子函数。

---

**建链阶段（MAC 层）**

**`start_advertising()`** — G 节点进入广播态，构建扩展广播帧。内部创建 `BroadcasterAccessManager` 并调用其 `build_ext_adv_frame()` 生成包含 AccessBasicInfo、SMF 参数的广播帧。失败返回 None。

```python
def start_advertising(self) -> BroadcastFrame | None:
    self._link_mgr.process_event(Event(EventType.START_BROADCAST))
    self._set_state(NodeState.ADVERTISING)
    self._broadcaster_mgr = BroadcasterAccessManager(
        link_manager=self._link_mgr,
        local_address=self.config.address,
        whitelist=self._whitelist)
    return self._broadcaster_mgr.build_ext_adv_frame()
```

其中 `BroadcasterAccessManager`（位于 `access.py`）负责管理广播阶段的接入资源、构建广播帧的各个 data item 字段。

---

**`start_scanning()`** — T 节点进入扫描态，监听广播帧。仅触发 LinkManager 状态事件，不返回数据。

```python
def start_scanning(self) -> None:
    self._link_mgr.process_event(Event(EventType.START_SCAN))
```

---

**`connect(peer_address)`** — T 节点向目标地址发起接入请求。设置本方 `peer_address`，通过 `SEND_ACCESS_REQUEST` 事件触发 LinkManager 进入接入流程，状态切换为 CONNECTING。

```python
def connect(self, peer_address: bytes) -> None:
    self._peer_address = peer_address
    self._link_mgr.process_event(Event(EventType.SEND_ACCESS_REQUEST))
    self._set_state(NodeState.CONNECTING)
```

---

**`accept_connection(peer_address, role)`** — G 节点接受连接。设置双方 peer_address，通过 `ACCESS_REQUEST_RECEIVED` 事件完成接入握手并分配 G/T 角色，状态切换为 CONNECTED。最后调用 `_register_link_schedule(link_id=0)` 在调度器中注册该链路的时间片资源。

```python
def accept_connection(self, peer_address: bytes, role: Role = Role.G_NODE) -> None:
    self._peer_address = peer_address
    self._link_mgr.peer_address = peer_address
    self._link_mgr.process_event(
        Event(EventType.ACCESS_REQUEST_RECEIVED, {"accepted": True, "role": role})
    )
    self._set_state(NodeState.CONNECTED)
    self._register_link_schedule(link_id=0)
```

其中 `_register_link_schedule()` 向 `ScheduleManager` 注册该链路的事件组和时间片，是后续 `transmit()` 能正常返回 IQ 的前提——没有调度资源则 `prepare_tx()` 返回空。

---

**数据传输阶段（链路层 → 物理层）**

**`send(payload)`** — 将数据提交到 QoS 发送队列。内部调用 `_qos.submit_data()` 创建 `TxQueueItem` 并 push 到 `tx_queue`，受流控（FlowController）约束——队列满则 push 失败返回 False。

```python
def send(self, data: bytes, priority: Priority = Priority.NORMAL) -> bool:
    return self._qos.submit_data(data, priority)
```

---

**`transmit()`** — 核心发送函数。从队列取帧 → 加密（可选）→ 组 MAC 帧 → `mac_to_iq()` 生成 IQ → 计算跳频信道 → 返回 `TxResult`。

```python
def transmit(self) -> TxResult:
    decision, item = self._qos.prepare_tx()        # 从队列取帧, 受流控 + ARQ 约束
    if item is None:                                 # 队列空或 ARQ 锁定 → 无帧可发
        return TxResult(iq=None, mac_bytes=None, decision=decision)
    payload = item.data
    # 加密 (如果启用)
    if self._crypto is not None and self.config.enable_encryption:
        ciphertext, mic = self._crypto.encrypt(payload, b"")
        payload = ciphertext + mic
    frame = AsyncDataFrame(segment_type=0, data=payload)
    mac_bytes = frame.pack()                         # MAC 帧序列化为字节
    ctrl_fields = self._qos.get_ctrl_fields()
    ctrl_info = _build_ctrl_info(ctrl_fields, len(mac_bytes))
    iq = mac_to_iq(mac_bytes, self._tx_config,       # MAC → PHY: 编码 + 调制 → IQ
                   ctrl_info=ctrl_info)
    # 跳频: 计算当前信道号 (基带仿真中仅记录, 不切换载波)
    slot = self._scheduler.slot_counter.value
    channel = data_link_hop(slot, self._hop_param2, self._freq_table)
    self._tx_count += 1
    return TxResult(iq=iq, mac_bytes=mac_bytes, ...)
```

其中嵌套的关键子函数：
- `_qos.prepare_tx()` — 检查 ARQ 状态（是否在等待 ACK）→ 从 `tx_queue` pop 一帧 → `flow.dequeue()` 更新流控计数器。返回 TxDecision + TxQueueItem。
- `mac_to_iq(mac_bytes, tx_config, ctrl_info)` — 完整的 MAC→PHY 发射管线：加 CRC → Polar 编码 → 速率匹配 → 加扰 → QPSK 调制 → RRC 脉冲成型 → 组帧头 → IQ 信号。
- `data_link_hop(slot, hop_param2, freq_table)` — PRNG 跳频序列生成，返回当前物理信道号（基带仿真中仅记录，实际频率切换在射频端完成）。

---

**`receive(iq, n_mac_bytes)`** — 核心接收函数。应用信道效应 → `iq_to_mac()` 解调解码 → 拆 MAC 帧取数据 → CRC 校验 → 返回 `RxResult`。

```python
def receive(self, iq_signal: np.ndarray, n_mac_bytes: int) -> RxResult:
    # 可选信道效应 (NodeConfig 中可预配 ChannelModel)
    if self._channel is not None:
        iq_signal = self._channel.apply_fading(iq_signal, self._tx_config.sps)
    # PHY → MAC: 同步 + 解调 + 译码
    rx: MacRxResult = iq_to_mac(iq_signal, self._tx_config, n_mac_bytes)
    if not rx.crc_ok:
        return RxResult(data=None, success=False)        # CRC 失败 → 丢弃
    # 拆帧取数据
    recovered = AsyncDataFrame.unpack(rx.mac_payload)
    data = recovered.data
    # 解密 (如果启用)
    if self._crypto is not None and self.config.enable_encryption:
        data = self._crypto.decrypt(data[:-4], data[-4:], b"")
    return RxResult(data=data, success=True)
```

其中嵌套的关键子函数：
- `iq_to_mac(iq, tx_config, n_mac_bytes)` — 完整的 PHY→MAC 接收管线：同步 → 解帧头 → 匹配滤波 → 解调 → 解扰 → 速率解匹配 → Polar 译码 → CRC 校验 → 返回 MacRxResult（含 `crc_ok` 和 `mac_payload`）。
- `AsyncDataFrame.unpack(mac_bytes)` — 将 MAC 字节流解析为帧对象，提取 `segment_type` 和 `data` 字段。

---

**信道阶段（物理层）**

- **`ChannelModel(snr_db)`** — 创建 AWGN 信道模型，内部封装 SNR 和噪声方差计算。
- **`apply_awgn(iq, sps)`** — 测量信号功率 → 根据 SNR 计算噪声方差 → 生成复高斯噪声叠加，原理同 01.02。`sps` 补偿上采样功率摊薄。

---

### 2. 接入成功率 SNR 扫描

在每个 SNR 下重复 100 次独立接入试验，分别统计广播成功率和数据往返成功率：

In [ ]:
from nearlink_sdr.mac.frame import AsyncDataFrame
from nearlink_sdr.phy.channel import ChannelModel

# ===== 扫描参数 =====
snr_range = np.arange(-4, 12, 2)    # SNR 范围 -4~10 dB
n_trials = 100                       # 每个 SNR 独立试验 100 次
rng = np.random.default_rng(42)

print(f"{'SNR':>5s}  {'Broadcast':>10s}  {'Roundtrip':>10s}  {'Overall':>10s}")
print("-" * 39)
results = []

for snr in snr_range:
    bc_ok = rt_ok = all_ok = 0       # 各阶段成功计数

    for _ in range(n_trials):
        # ---- 创建 G/T 节点 ----
        g = SleNode(config=NodeConfig(
            address=g_addr, role=NodeRole.G_NODE,
            frame_type=2, mcs_index=0, max_retransmit=0))   # MCS=0: BPSK 1/4, 最鲁棒
        t = SleNode(config=NodeConfig(
            address=t_addr, role=NodeRole.T_NODE,
            frame_type=2, mcs_index=0, max_retransmit=0))

        # ---- 阶段 (a): G 发送广播帧 ----
        adv_frame = g.start_advertising()                    # 生成广播帧
        if not adv_frame:                                    # 广播帧无效 → 跳过
            continue
        bc_ok += 1                                           # 广播成功 +1

        # ---- 阶段 (b)-(d): T 扫描, 发起连接, G 接受 ----
        t.start_scanning()                                   # T 开始扫描
        t.connect(g_addr)                                    # T 发起连接请求
        g.accept_connection(t_addr, Role.G_NODE)             # G 接受, 链路建立

        # ---- 阶段 (e): 数据交换 (通过 AWGN 信道) ----
        payload = b"test_data"
        g.send(payload)                                      # G 将数据压入发送队列
        tx = g.transmit()                                    # G 生成 IQ 信号
        if tx.iq is None:                                    # IQ 生成失败 → 跳过
            continue

        ch = ChannelModel(snr_db=float(snr))                 # 创建 AWGN 信道
        noisy_iq = ch.apply_awgn(tx.iq, g._tx_config.sps)    # IQ 信号加噪
        frame = AsyncDataFrame(segment_type=0, data=payload) # 构造期望的 MAC 帧
        rx = t.receive(noisy_iq, len(frame.pack()))          # T 接收并解调解码
        if rx.success and rx.data == payload:                # CRC 通过 且 数据匹配
            rt_ok += 1

        # ---- 阶段 (f): 断开连接 ----
        scheduler_ok = 0 in g.scheduler.event_schedulers     # 确认调度器已注册链路
        g.disconnect()                                        # G 发起断开
        # 整体成功: 广播OK + 数据往返OK + 调度器OK + 已断开
        if all([bc_ok, rx.success and rx.data == payload, scheduler_ok,
                g.state == NodeState.DISCONNECTED]):
            all_ok += 1

    results.append((snr, bc_ok, rt_ok, all_ok))              # 保存该 SNR 结果
    print(f"{snr:5.0f}  {bc_ok/n_trials:10.4f}  {rt_ok/n_trials:10.4f}  {all_ok/n_trials:10.4f}")

### 3. 成功率曲线

In [ ]:
import matplotlib.pyplot as plt

snrs = [r[0] for r in results]
rt = [r[2] / n_trials for r in results]
ov = [r[3] / n_trials for r in results]

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(snrs, ov, "s-", lw=2, label="Overall")
ax.plot(snrs, rt, "o--", lw=2, label="Data Roundtrip")
ax.axhline(y=0.95, color="gray", ls="--", alpha=0.5, label="95% threshold")
ax.set_xlabel("SNR (dB)"); ax.set_ylabel("Success Rate")
ax.set_title("Access Flow Success Rate (MCS=0, AWGN)")
ax.legend(); ax.grid(True, ls="--", alpha=0.5); ax.set_ylim(0, 1.05)
plt.show()

idx95 = next(i for i, v in enumerate(ov) if v >= 0.95)

## 课后实践

请补全下方接入建链与数据交换流程中的 **3 处空缺**（每处一行代码），完成 G/T 节点建链并通过 AWGN 信道验证数据往返。

要求：

1. **建链（1 处）**：补全 G 节点接受连接的关键调用
2. **信道（1 处）**：补全 AWGN 加噪
3. **验证（1 处）**：补全数据往返成功的判断条件

完成后运行 ，观察建链状态和数据往返结果。

In [ ]:
%%writefile access_practice.py
import sys
sys.path.insert(0, "../src")
import numpy as np
from nearlink_sdr.mac.frame import AsyncDataFrame
from nearlink_sdr.mac.link_manager import Role
from nearlink_sdr.node import NodeConfig, NodeRole, SleNode
from nearlink_sdr.phy.channel import ChannelModel

g_addr = b"\x01\x02\x03\x04\x05\x06"
t_addr = b"\x0A\x0B\x0C\x0D\x0E\x0F"

snr_db = 10.0

# ===== 创建 G/T 节点 =====
g = SleNode(config=NodeConfig(
    address=g_addr, role=NodeRole.G_NODE,
    frame_type=2, mcs_index=0, max_retransmit=0))
t = SleNode(config=NodeConfig(
    address=t_addr, role=NodeRole.T_NODE,
    frame_type=2, mcs_index=0, max_retransmit=0))

# ===== 建链 =====
g.start_advertising()                             # G 开始广播
t.start_scanning()                                # T 开始扫描
t.connect(g_addr)                                 # T 发起连接
______________                                    # 1: G 接受连接 （补全）

linked = t.peer_address == g_addr and g.peer_address == t_addr
print(f"Link: {"OK" if linked else "FAIL"}  "
      f"G={g.state.name} T={t.state.name}")

# ===== 数据交换 =====
payload = b"practice_test"
g.send(payload)                                   # G 压入发送队列
tx = g.transmit()                                 # G 生成 IQ 信号

ch = ChannelModel(snr_db=snr_db)                  # AWGN 信道
______________                                    # 2: 对 IQ 信号加噪（补全）

frame = AsyncDataFrame(segment_type=0, data=payload)
rx = t.receive(noisy_iq, len(frame.pack()))      # T 接收并解调
______________                                    # 3: 判断数据往返成功（补全）
    print("Data roundtrip: OK")
else:
    print(f"Data roundtrip: FAIL (CRC={"OK" if rx.crc_ok else "FAIL"})")

g.disconnect()
print(f"Final G state: {g.state.name}")


执行以下命令进行编译并验证结果：


In [ ]:
!python access_practice.py


执行以下代码获取答案


In [ ]:
!cat answer/05.04_answer.txt
